# Engine

## Steps

1. Load hazard, exposure
2. Inside Engine (because of caching:)
   1. Reproject hazard onto exposure (maybe cache)
   2. *later:* Resample exposure to hazard non-spatial dimensions (maybe cache)
   3. *optional:* Apply unsequa samples
   4. Regional cutouts. Single, multiple
   5. Impact calculation (maybe cache)
   6. Compute aggregates on total thing
      1. `impact_at_reg` is just `at_event` for each cutout
      2. Place aggregates in datatree with empty root

## Caching

* Is data chunked?
  * Yes: Write cache
  * No: Data is transformed in memory. Do not cache.
* Cache before computing impact

In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import pandas as pd
import xarray as xr

# Single exposure array with time axis
exposure = xr.concat(
    [
        xr.open_dataarray(
            "data/exposure/deu_ppp_2000_UNadj.tif",
            chunks="auto",
            decode_coords="all",
            engine="rasterio",
        ),
        xr.open_dataarray(
            "data/exposure/deu_ppp_2001_UNadj.tif",
            chunks="auto",
            decode_coords="all",
            engine="rasterio",
        ),
    ],
    dim=pd.DatetimeIndex(["2021-06-12", "2021-06-15"], name="time"),
)
exposure = exposure.squeeze()  # band

# exposure = xr.open_dataarray(
#     "data/exposure/deu_ppp_2000_UNadj.tif",
#     chunks="auto",
#     decode_coords="all",
#     engine="rasterio",
# ).squeeze()
# exposure = (
#     exposure.sel(x=slice(5.88, 5.89), y=slice(55, 54.99)).chunk("auto")
# )
# exposure = exposure.rename({"x": "longitude", "y": "latitude"})
exposure

<xarray.DataArray 'band_data' (time: 2, y: 9345, x: 10999)> Size: 822MB
dask.array<getitem, shape=(2, 9345, 10999), dtype=float32, chunksize=(1, 2560, 10999), chunktype=numpy.ndarray>
Coordinates:
    band         int64 8B 1
  * x            (x) float64 88kB 5.872 5.873 5.874 5.875 ... 15.04 15.04 15.04
  * y            (y) float64 75kB 55.06 55.06 55.05 55.05 ... 47.27 47.27 47.27
    spatial_ref  int64 8B 0
  * time         (time) datetime64[ns] 16B 2021-06-12 2021-06-15
Attributes:
    AREA_OR_POINT:  Area

In [3]:
from functools import partial
import numpy as np


def impact_func(intensity, threshold, half_point, exponent, scale):
    luk = (intensity - threshold) / (half_point - threshold)
    # luk = np.where(luk < 0, 0, luk)
    return np.maximum(scale * luk**exponent / (1 + luk**exponent), 0.0)


# impf = partial(impact_func, threshold=0.1, half_point=1.0, scale=1.0, exponent=3)
impf = lambda intensity: impact_func(
    intensity, threshold=0.1, half_point=1.0, scale=1.0, exponent=3
)

In [7]:
import odc.geo.xr  # noqa: F401

hazard = xr.open_dataset(
    "data/hazard/flood-2021-06-fine.nc", chunks="auto", decode_coords="all"
)["flood_depth"]
hazard = hazard.odc.assign_crs("EPSG:4326")
# hazard = hazard.isel(time=0, drop=True)
# hazard = hazard.sel(time=exposure["time"])
# hazard = hazard.sel(longitude=slice(5.88, 5.89), latitude=slice(55, 54.99)).isel(number=0, time=0, step=0).chunk("auto")

# To dataset
hazard

<xarray.DataArray 'flood_depth' (latitude: 1859, longitude: 2183, number: 10,
                                 time: 3, step: 5)> Size: 2GB
dask.array<open_dataset-flood_depth, shape=(1859, 2183, 10, 3, 5), dtype=float32, chunksize=(620, 728, 4, 1, 1), chunktype=numpy.ndarray>
Coordinates:
  * number       (number) int64 80B 1 2 3 4 5 6 7 8 9 10
  * time         (time) datetime64[ns] 24B 2021-06-12 2021-06-13 2021-06-15
  * step         (step) timedelta64[ns] 40B 1 days 2 days 3 days 4 days 5 days
    surface      float64 8B ...
    valid_time   (time, step) datetime64[ns] 120B dask.array<chunksize=(3, 5), meta=np.ndarray>
  * longitude    (longitude) float64 17kB 5.878 5.883 5.887 ... 14.97 14.97
  * latitude     (latitude) float64 15kB 55.02 55.02 55.01 ... 47.29 47.28 47.28
    spatial_ref  int32 4B 4326

In [6]:
from climadace.funcs import reproject_hazard

hazard = reproject_hazard(hazard, exposure)
hazard

/Users/ldr.riedel/miniforge3/envs/climada_env_3.11/lib/python3.11/site-packages/dask/array/core.py:4988: PerformanceWarning: Increasing number of chunks by factor of 15
  result = blockwise(


NOT CACHING!


<xarray.DataArray 'flood_depth' (y: 9345, x: 10999, number: 10, time: 3, step: 5)> Size: 62GB
dask.array<rechunk-merge, shape=(9345, 10999, 10, 3, 5), dtype=float32, chunksize=(2560, 10999, 1, 1, 1), chunktype=numpy.ndarray>
Coordinates:
  * number       (number) int64 80B 1 2 3 4 5 6 7 8 9 10
  * time         (time) datetime64[ns] 24B 2021-06-12 2021-06-13 2021-06-15
  * step         (step) timedelta64[ns] 40B 1 days 2 days 3 days 4 days 5 days
  * x            (x) float64 88kB 5.872 5.873 5.874 5.875 ... 15.04 15.04 15.04
  * y            (y) float64 75kB 55.06 55.06 55.05 55.05 ... 47.27 47.27 47.27
    surface      float64 8B ...
    valid_time   (time, step) datetime64[ns] 120B dask.array<chunksize=(1, 1), meta=np.ndarray>
    spatial_ref  int32 4B 4326

In [ ]:
from climadace.funcs import align_events

exposure = align_events(hazard, exposure, "time")
exposure

<xarray.DataArray 'band_data' (time: 3, y: 9345, x: 10999)> Size: 1GB
dask.array<getitem, shape=(3, 9345, 10999), dtype=float32, chunksize=(1, 2560, 10999), chunktype=numpy.ndarray>
Coordinates:
    band         int64 8B 1
  * x            (x) float64 88kB 5.872 5.873 5.874 5.875 ... 15.04 15.04 15.04
  * y            (y) float64 75kB 55.06 55.06 55.05 55.05 ... 47.27 47.27 47.27
    spatial_ref  int64 8B 0
  * time         (time) datetime64[ns] 24B 2021-06-12 2021-06-12 2021-06-15
Attributes:
    AREA_OR_POINT:  Area

: 

: 

: 

: 

In [ ]:
from climadace.engine import EngineOld

engine = EngineOld(hazard, exposure, impf)

: 

: 

: 

: 

In [ ]:
engine.reproject_hazard()

/Users/ldr.riedel/miniforge3/envs/climada_env_3.11/lib/python3.11/site-packages/dask/array/core.py:4988: PerformanceWarning: Increasing number of chunks by factor of 15
  result = blockwise(


NOT CACHING!


<xarray.DataArray 'flood_depth' (y: 9345, x: 10999, number: 10, time: 2, step: 5)> Size: 41GB
dask.array<open_dataset-flood_depth, shape=(9345, 10999, 10, 2, 5), dtype=float32, chunksize=(2560, 10999, 1, 1, 1), chunktype=numpy.ndarray>
Coordinates:
    band         int64 8B ...
  * number       (number) int64 80B 1 2 3 4 5 6 7 8 9 10
    spatial_ref  int32 4B ...
  * step         (step) timedelta64[ns] 40B 1 days 2 days 3 days 4 days 5 days
    surface      float64 8B ...
  * time         (time) datetime64[ns] 16B 2021-06-12 2021-06-15
    valid_time   (time, step) datetime64[ns] 80B dask.array<chunksize=(2, 5), meta=np.ndarray>
  * x            (x) float64 88kB 5.872 5.873 5.874 5.875 ... 15.04 15.04 15.04
  * y            (y) float64 75kB 55.06 55.06 55.05 55.05 ... 47.27 47.27 47.27

: 

: 

: 

: 

In [ ]:
engine.compute()

<xarray.DataArray (y: 9345, x: 10999, number: 10, time: 2, step: 5)> Size: 41GB
dask.array<open_dataset-__xarray_dataarray_variable__, shape=(9345, 10999, 10, 2, 5), dtype=float32, chunksize=(2560, 10999, 1, 1, 1), chunktype=numpy.ndarray>
Coordinates:
    band        int64 8B ...
  * number      (number) int64 80B 1 2 3 4 5 6 7 8 9 10
  * step        (step) timedelta64[ns] 40B 1 days 2 days 3 days 4 days 5 days
    surface     float64 8B ...
  * time        (time) datetime64[ns] 16B 2021-06-12 2021-06-15
    valid_time  (time, step) datetime64[ns] 80B dask.array<chunksize=(2, 5), meta=np.ndarray>
  * x           (x) float64 88kB 5.872 5.873 5.874 5.875 ... 15.04 15.04 15.04
  * y           (y) float64 75kB 55.06 55.06 55.05 55.05 ... 47.27 47.27 47.27

: 

: 

: 

: 

In [ ]:
import climadace

engine.event_dims = climadace.engine.derive_event_dims(engine.hazard)
engine.event_dims

{'time': 'step', 'event': 'step'}

: 

: 

: 

: 

In [ ]:
from dask.distributed import Client

with Client(n_workers=1, threads_per_worker=2, memory_limit="4GB"):
    engine.aggregate()

AttributeError: 'DataArray' object has no attribute 'data_vars'

: 

: 

: 

: 

In [ ]:
engine.aggregate()

/Users/ldr.riedel/miniforge3/envs/climada_env_3.12/lib/python3.12/site-packages/zarr/api/asynchronous.py:213: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
<string>:9: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.


<xarray.Dataset> Size: 1kB
Dimensions:                (number: 10, time: 2, step: 5, year: 1)
Coordinates:
    valid_time             (time, step) datetime64[ns] 80B dask.array<chunksize=(2, 5), meta=np.ndarray>
    surface                float64 8B ...
    band                   int64 8B ...
  * year                   (year) int64 8B 2021
  * step                   (step) timedelta64[ns] 40B 1 days 2 days ... 5 days
  * number                 (number) int64 80B 1 2 3 4 5 6 7 8 9 10
  * time                   (time) datetime64[ns] 16B 2021-06-12 2021-06-15
Data variables:
    at_event               (number, time, step) float32 400B dask.array<chunksize=(10, 2, 5), meta=np.ndarray>
    average_annual_impact  (number, year, step) float32 200B dask.array<chunksize=(10, 1, 5), meta=np.ndarray>
    average_impact         (number, step) float32 200B dask.array<chunksize=(10, 5), meta=np.ndarray>

: 

: 

: 

: 

In [ ]:
engine.aggregates

<xarray.Dataset> Size: 1kB
Dimensions:                (number: 10, time: 2, step: 5, year: 1)
Coordinates:
    band                   int64 8B ...
    surface                float64 8B ...
    valid_time             (time, step) datetime64[ns] 80B dask.array<chunksize=(2, 5), meta=np.ndarray>
  * time                   (time) datetime64[ns] 16B 2021-06-12 2021-06-15
  * year                   (year) int64 8B 2021
  * step                   (step) timedelta64[ns] 40B 1 days 2 days ... 5 days
  * number                 (number) int64 80B 1 2 3 4 5 6 7 8 9 10
Data variables:
    at_event               (number, time, step) float32 400B dask.array<chunksize=(10, 2, 5), meta=np.ndarray>
    average_annual_impact  (number, year, step) float32 200B dask.array<chunksize=(10, 1, 5), meta=np.ndarray>
    average_impact         (number, step) float32 200B dask.array<chunksize=(10, 5), meta=np.ndarray>

: 

: 

: 

: 

In [ ]:
engine.aggregates["average_annual_impact"]

<xarray.DataArray 'average_annual_impact' (number: 10, year: 1, step: 5)> Size: 200B
dask.array<open_dataset-average_annual_impact, shape=(10, 1, 5), dtype=float32, chunksize=(10, 1, 5), chunktype=numpy.ndarray>
Coordinates:
    band     int64 8B ...
    surface  float64 8B ...
  * year     (year) int64 8B 2021
  * step     (step) timedelta64[ns] 40B 1 days 2 days 3 days 4 days 5 days
  * number   (number) int64 80B 1 2 3 4 5 6 7 8 9 10

: 

: 

: 

: 